# Upload consolidated OMR dataset to Hugging Face

Uploads your training `consolidated.zip` (~139k image / PrIMuS semantic pairs) to dataset repo **`NathanZ721/omr-monophonic-consolidated`**.

## Before you run

1. Hugging Face **Write** token: https://huggingface.co/settings/tokens
2. Put `consolidated.zip` on Drive at `MyDrive/omr/consolidated.zip`  
   (same zip your training notebooks already use; build with `scripts/build_consolidated_zip.py` if needed)
3. CPU runtime is enough

The notebook uploads:
- `consolidated.zip` — full archive
- `README.md` — dataset card

In [ ]:
# @title 1) Mount Drive and configure
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

HF_TOKEN = ""  # @param {type:"string"}
HF_REPO_ID = "NathanZ721/omr-monophonic-consolidated"  # @param {type:"string"}
HF_REPO_PRIVATE = False  # @param {type:"boolean"}

DRIVE_ZIP = Path("/content/drive/MyDrive/omr/consolidated.zip")
LOCAL_CARD = Path("/content/dataset_README.md")

if not HF_TOKEN.strip():
    raise ValueError("Set HF_TOKEN to a Hugging Face Write token.")
if "/" not in HF_REPO_ID:
    raise ValueError("HF_REPO_ID must look like username/repo-name")
if not DRIVE_ZIP.is_file():
    raise FileNotFoundError(
        f"Missing {DRIVE_ZIP}\n"
        "Upload consolidated.zip to MyDrive/omr/ first "
        "(or build it with scripts/build_consolidated_zip.py)."
    )

size_gb = DRIVE_ZIP.stat().st_size / (1024 ** 3)
print(f"Zip: {DRIVE_ZIP}  ({size_gb:.2f} GB)")
print(f"Repo: {HF_REPO_ID}  private={HF_REPO_PRIVATE}")

In [ ]:
# @title 2) Install huggingface_hub
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"])
print("huggingface_hub ready")

In [ ]:
# @title 3) Write dataset card
from pathlib import Path

LOCAL_CARD = Path("/content/dataset_README.md")
HF_REPO_ID = "NathanZ721/omr-monophonic-consolidated"  # @param {type:"string"}

LOCAL_CARD.write_text(
    f"""---
pretty_name: OMR Monophonic Consolidated
license: apache-2.0
task_categories:
- image-to-text
tags:
- optical-music-recognition
- omr
- primus
- sheet-music
- monophonic
size_categories:
- 100K<n<1M
---

# OMR Monophonic Consolidated

Paired monophonic sheet-music images and **PrIMuS semantic** ground-truth strings used to fine-tune [`NathanZ721/omr-monophonic`](https://huggingface.co/NathanZ721/omr-monophonic).

## Contents

Download `consolidated.zip`. After extraction:

```text
consolidated/
  images/
  ground_truth/
```

Filenames are `{{group}}__{{id}}.{{ext}}` with matching `.semantic` siblings.

### Groups (~139k pairs)

| Group | Approx. count | Description |
| --- | ---: | --- |
| `primus` | 36,951 | Clean digital PrIMuS renders |
| `camera_primus` | 36,951 | Camera-style distorted PrIMuS |
| `mashcima` | 36,584 | Handwritten Mashcima synthesis |
| `corrupt_clean` | 9,806 | Corrupted clean digital |
| `corrupt_camera` | 9,734 | Corrupted camera-distorted |
| `corrupt_handwritten` | 9,456 | Corrupted handwritten |

## Related model

- [`NathanZ721/omr-monophonic`](https://huggingface.co/NathanZ721/omr-monophonic)

## Load

```python
from huggingface_hub import hf_hub_download
import zipfile
from pathlib import Path

zip_path = hf_hub_download(
    repo_id="{HF_REPO_ID}",
    filename="consolidated.zip",
    repo_type="dataset",
)
out = Path("/content/omr_data")
out.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(out)
```
""",
    encoding="utf-8",
)
print(f"Wrote {LOCAL_CARD}")

In [ ]:
# @title 4) Create dataset repo and upload zip + README
from pathlib import Path

from huggingface_hub import HfApi, create_repo, upload_file

HF_TOKEN = ""  # @param {type:"string"}
HF_REPO_ID = "NathanZ721/omr-monophonic-consolidated"  # @param {type:"string"}
HF_REPO_PRIVATE = False  # @param {type:"boolean"}
DRIVE_ZIP = Path("/content/drive/MyDrive/omr/consolidated.zip")
LOCAL_CARD = Path("/content/dataset_README.md")

api = HfApi(token=HF_TOKEN)
user = api.whoami()
print(f"Authenticated as: {user.get('name', user)}")

create_repo(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    private=HF_REPO_PRIVATE,
    exist_ok=True,
    token=HF_TOKEN,
)
print(f"Dataset repo ready: https://huggingface.co/datasets/{HF_REPO_ID}")

print("Uploading README.md …")
upload_file(
    path_or_fileobj=str(LOCAL_CARD),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    token=HF_TOKEN,
    commit_message="Add dataset card",
)

print(f"Uploading consolidated.zip ({DRIVE_ZIP.stat().st_size / (1024**3):.2f} GB) …")
print("This can take a while on a large archive; leave the runtime connected.")
upload_file(
    path_or_fileobj=str(DRIVE_ZIP),
    path_in_repo="consolidated.zip",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    token=HF_TOKEN,
    commit_message="Add consolidated.zip (~139k image/semantic pairs)",
)

print("=" * 72)
print(f"Done: https://huggingface.co/datasets/{HF_REPO_ID}")
print("Files: README.md, consolidated.zip")
print("=" * 72)